# GazeOS: cloud training (Colab or Kaggle)

Runs any list of `training.train` jobs on a cloud GPU (docs/PLAN.md section 3).

- **Colab:** shards and outputs live in your Google Drive. Outputs are written straight to Drive, so a
  disconnect loses at most the current epoch: re-run the notebook and every job resumes from `last.pt`.
- **Kaggle:** add the shards as a private Kaggle dataset (or download them from Drive in cell 3);
  outputs go to `/kaggle/working`. Save a notebook version before the 12-hour limit to keep them.

Rules: only packed shards go to the cloud, never raw datasets; keep the Drive folder and Kaggle dataset
private (the datasets are non-commercial research data).

## 1. Settings: edit this cell

In [ ]:
import os
from pathlib import Path

PLATFORM = "kaggle" if os.path.exists("/kaggle") else "colab"

REPO_URL = "https://github.com/deep-9549/Gaze-Os.git"
BRANCH = "feature/no-ethxgaze-pipeline"
# Private repo: store a GitHub token as the secret GITHUB_TOKEN (Colab: key icon; Kaggle: Add-ons > Secrets).

# Folder holding shards/ and weights/ exactly as on the laptop (F:\GazeOS-data).
DRIVE_DATA = "/content/drive/MyDrive/Minor Project/GazeOS/data"   # Colab (G:\My Drive\Minor Project on the laptop)
KAGGLE_DATA = "/kaggle/input/gazeos-shards"              # Kaggle dataset mount

# Each job: (name, Hydra overrides). Jobs that already finished are skipped; unfinished ones resume.
JOBS = [
    ("stage_b_eth_fold0", ["experiment=stage_b_eth_init", "data.fold=0"]),
    # ("stage_a_rtgene", ["experiment=stage_a_rtgene"]),
    # *[(f"stage_b_fold{f}_seed{s}", ["experiment=stage_b_mpii", f"data.fold={f}", f"seed={s}",
    #     "init_checkpoint=outputs/stage_a_rtgene/best.pt"]) for f in range(15) for s in range(3)],
]
SHARD_SETS = ["mpiifacegaze"]          # which data/shards/<name> folders the jobs need

## 2. Code and dependencies

In [ ]:
import subprocess

def secret(name):
    try:
        if PLATFORM == "colab":
            from google.colab import userdata
            return userdata.get(name)
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

WORK = Path("/content" if PLATFORM == "colab" else "/kaggle/working")
REPO = WORK / "Gaze-Os"
if not REPO.exists():
    token = secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
os.chdir(REPO)
subprocess.run(["pip", "install", "-q", "-e", ".[train]"], check=True)

## 3. Data: copy shards to the local disk (HDF5 over the Drive mount is too slow to train from)

In [ ]:
import shutil

if PLATFORM == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    src = Path(DRIVE_DATA)
    OUT = src.parent / "outputs"            # checkpoints on Drive survive disconnects
else:
    src = Path(KAGGLE_DATA)
    OUT = WORK / "outputs"
local = Path("/tmp/gazeos-data")
for name in SHARD_SETS:
    dst = local / "shards" / name
    if not dst.exists():
        shutil.copytree(src / "shards" / name, dst)
    print(name, len(list(dst.glob("*.h5"))), "shards")
if (src / "weights").exists() and not (local / "weights").exists():
    shutil.copytree(src / "weights", local / "weights")
OUT.mkdir(parents=True, exist_ok=True)
# Configs refer to data/... and outputs/...; point both at the right places.
for link, target in (("data", local), ("outputs", OUT)):
    p = REPO / link
    if not p.exists():
        p.symlink_to(target, target_is_directory=True)

## 4. Train

In [ ]:
import json
import sys

import torch

print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU: enable one in the runtime settings")
for name, overrides in JOBS:
    out = OUT / name
    if (out / "result.json").exists():
        print("done already:", name)
        continue
    cmd = [sys.executable, "-m", "training.train", *overrides, f"output_dir=outputs/{name}",
           "train.resume=true", "data.workers=2"]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)

## 5. Results

In [ ]:
for name, _ in JOBS:
    f = OUT / name / "result.json"
    if f.exists():
        r = json.loads(f.read_text())
        test = r.get("test", {}).get("mean_error_deg")
        print(f"{name:30s} best val {r['best_val']:.2f}  test {test if test is None else round(test, 2)}")